# 02_ROCOv2_Dataset_Preparation

**Purpose:** Create the fixed, leakage-controlled experimental metadata used by all downstream notebooks.

This notebook reads the original ROCOv2 split metadata and image ZIP archives, verifies integrity, constructs the verified 18-CUI taxonomy, creates multi-hot targets, and writes lightweight prepared CSV files. **Image ZIP archives are never copied or modified.**

### Fixed protocol
- Train → model training
- Validation → calibration/development/model-selection split
- Test → untouched final evaluation split
- No random repartitioning is performed here or later.
- `label_mapping.json` is the authoritative label order.


In [1]:

# ============================================================
# CELL 1 — CONFIGURATION AND REPRODUCIBILITY
# ============================================================

import os
import sys
import json
import random
import platform
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Google Colab / Google Drive
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    IN_COLAB = False

# Explicit paths
BASE_DIR = Path("/content/drive/MyDrive/ROCOv2")
PREPARED_DIR = BASE_DIR / "Prepared"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
RARE_LABEL_THRESHOLD = 10

RAW_FILES = {
    "Train": {
        "captions": BASE_DIR / "Train" / "Captions" / "train_captions.csv",
        "concepts": BASE_DIR / "Train" / "Concepts" / "train_concepts.csv",
        "images": BASE_DIR / "Train" / "Images" / "train_images.zip",
    },
    "Validation": {
        "captions": BASE_DIR / "Validation" / "Captions" / "valid_captions.csv",
        "concepts": BASE_DIR / "Validation" / "Concepts" / "valid_concepts.csv",
        "images": BASE_DIR / "Validation" / "Images" / "valid_images.zip",
    },
    "Test": {
        "captions": BASE_DIR / "Test" / "Captions" / "test_captions.csv",
        "concepts": BASE_DIR / "Test" / "Concepts" / "test_concepts.csv",
        "images": BASE_DIR / "Test" / "Images" / "test_images.zip",
    },
}

CUI_MAPPING_PATH = BASE_DIR / "cui_mapping.csv"

SPLIT_PROTOCOL = {
    "Train": "training",
    "Validation": "validation/calibration-development",
    "Test": "untouched final evaluation",
}

print("IN_COLAB:", IN_COLAB)
print("BASE_DIR:", BASE_DIR)
print("PREPARED_DIR:", PREPARED_DIR)
print("SEED:", SEED)


Mounted at /content/drive
IN_COLAB: True
BASE_DIR: /content/drive/MyDrive/ROCOv2
PREPARED_DIR: /content/drive/MyDrive/ROCOv2/Prepared
SEED: 42


In [2]:

# ============================================================
# CELL 2 — SET SEEDS AND RECORD ENVIRONMENT
# ============================================================

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_global_seed(SEED)

def package_version(module_name):
    try:
        module = __import__(module_name)
        return getattr(module, "__version__", "unknown")
    except Exception:
        return "not installed"

try:
    import torch
    torch_version = torch.__version__
    cuda_version = torch.version.cuda
    gpu_name = (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    )
except Exception:
    torch_version = "not installed"
    cuda_version = None
    gpu_name = None

ENVIRONMENT = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "pytorch_version": torch_version,
    "torchvision_version": package_version("torchvision"),
    "transformers_version": package_version("transformers"),
    "sklearn_version": package_version("sklearn"),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "cuda_version": cuda_version,
    "gpu_name": gpu_name,
    "random_seed": SEED,
}

display(pd.DataFrame(
    list(ENVIRONMENT.items()),
    columns=["item", "value"]
))


,item,value
0,python_version,"3.12.13 (main, Mar 4 2026, 09:23:07) [GCC 11...."
1,platform,Linux-6.6.122+-x86_64-with-glibc2.35
2,pytorch_version,2.11.0+cu128
3,torchvision_version,0.26.0+cu128
4,transformers_version,5.15.0
5,sklearn_version,1.6.1
6,numpy_version,2.0.2
7,pandas_version,2.2.3
8,cuda_version,12.8
9,gpu_name,Tesla T4


In [3]:

# ============================================================
# CELL 3 — ASSERT REQUIRED ORIGINAL INPUTS EXIST
# ============================================================

required_paths = [CUI_MAPPING_PATH]

for split, files in RAW_FILES.items():
    required_paths.extend(files.values())

missing_files = [
    str(path) for path in required_paths
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required original ROCOv2 files are missing:\n\n"
        + "\n".join(missing_files)
        + "\n\n"
        "Restore the original ROCOv2 dataset before running Notebook 02. "
        "Do not substitute processed metadata files."
    )

print("PASS — All required original ROCOv2 inputs exist.")


PASS — All required original ROCOv2 inputs exist.


## 2. Load original ROCOv2 metadata

In [4]:

# ============================================================
# CELL 4 — LOAD RAW METADATA
# ============================================================

captions = {}
concepts = {}

for split, files in RAW_FILES.items():
    captions[split] = pd.read_csv(files["captions"])
    concepts[split] = pd.read_csv(files["concepts"])

    print(
        f"{split:12s} | "
        f"captions={len(captions[split]):,} | "
        f"concepts={len(concepts[split]):,}"
    )

cui_mapping_df = pd.read_csv(CUI_MAPPING_PATH)

for split in RAW_FILES:
    if "ID" not in captions[split].columns:
        raise ValueError(
            f"{split} captions file must contain an 'ID' column."
        )
    if not {"ID", "CUIs"}.issubset(concepts[split].columns):
        raise ValueError(
            f"{split} concepts file must contain 'ID' and 'CUIs'."
        )

if not {"CUI", "Canonical name"}.issubset(cui_mapping_df.columns):
    raise ValueError(
        "cui_mapping.csv must contain 'CUI' and 'Canonical name'."
    )

print(f"CUI mapping rows: {len(cui_mapping_df):,}")


Train        | captions=59,958 | concepts=59,958
Validation   | captions=9,904 | concepts=9,904
Test         | captions=9,927 | concepts=9,927
CUI mapping rows: 1,934


## 3. Verify fixed train/validation/test split

In [5]:

# ============================================================
# CELL 5 — FIXED SPLIT VERIFICATION
# ============================================================

for split in RAW_FILES:
    caption_ids = set(
        captions[split]["ID"].astype(str).str.strip()
    )
    concept_ids = set(
        concepts[split]["ID"].astype(str).str.strip()
    )

    if caption_ids != concept_ids:
        raise AssertionError(
            f"{split}: caption/concept ID sets differ.\n"
            f"Only in captions: {sorted(caption_ids - concept_ids)[:10]}\n"
            f"Only in concepts: {sorted(concept_ids - caption_ids)[:10]}"
        )

train_ids = set(captions["Train"]["ID"].astype(str).str.strip())
valid_ids = set(captions["Validation"]["ID"].astype(str).str.strip())
test_ids = set(captions["Test"]["ID"].astype(str).str.strip())

overlaps = {
    "Train_Validation": train_ids & valid_ids,
    "Train_Test": train_ids & test_ids,
    "Validation_Test": valid_ids & test_ids,
}

for pair, overlap in overlaps.items():
    if overlap:
        raise AssertionError(
            f"Split leakage detected: {pair} has {len(overlap)} "
            f"shared IDs. Examples: {list(overlap)[:10]}"
        )

split_statistics = pd.DataFrame([
    {
        "Split": split,
        "Records": len(captions[split]),
        "Unique_IDs": captions[split]["ID"].nunique(),
        "Protocol": SPLIT_PROTOCOL[split],
    }
    for split in ["Train", "Validation", "Test"]
])

display(split_statistics)
print("PASS — Fixed split verified; no random repartitioning performed.")


,Split,Records,Unique_IDs,Protocol
0,Train,59958,59958,training
1,Validation,9904,9904,validation/calibration-development
2,Test,9927,9927,untouched final evaluation


PASS — Fixed split verified; no random repartitioning performed.


## 4. Verify image files

In [6]:

# ============================================================
# CELL 6 — VERIFY IMAGES INSIDE ORIGINAL ZIP ARCHIVES
# ============================================================

def normalize_member(name):
    return str(name).replace("\\", "/").lstrip("./")

def build_zip_index(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = [
            normalize_member(n)
            for n in zf.namelist()
            if not n.endswith("/")
        ]

    by_stem = {}
    for member in members:
        stem = Path(member).stem
        by_stem.setdefault(stem, []).append(member)

    return set(members), by_stem

zip_indexes = {}

for split, files in RAW_FILES.items():
    names, by_stem = build_zip_index(files["images"])
    zip_indexes[split] = {
        "names": names,
        "by_stem": by_stem,
    }
    print(f"{split:12s}: {len(names):,} ZIP members")

def resolve_image_member(image_id, split):
    image_id = str(image_id).strip()
    index = zip_indexes[split]

    for candidate in [
        image_id,
        f"{image_id}.jpg",
        f"{image_id}.jpeg",
        f"{image_id}.png",
    ]:
        if candidate in index["names"]:
            return candidate

    matches = index["by_stem"].get(Path(image_id).stem, [])

    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise AssertionError(
            f"Ambiguous ZIP image match for {image_id} in {split}: "
            f"{matches[:10]}"
        )

    return None

image_audit_rows = []

for split in ["Train", "Validation", "Test"]:
    missing = []

    for image_id in captions[split]["ID"]:
        if resolve_image_member(image_id, split) is None:
            missing.append(str(image_id))

    image_audit_rows.append({
        "Split": split,
        "Metadata_Records": len(captions[split]),
        "Missing_Images": len(missing),
    })

    if missing:
        raise FileNotFoundError(
            f"{split}: {len(missing)} images referenced by metadata "
            f"are absent from its original ZIP. Examples: {missing[:20]}"
        )

image_audit_df = pd.DataFrame(image_audit_rows)
display(image_audit_df)

print("PASS — Every referenced image exists in the appropriate ZIP.")
print("No ZIP archive was copied or modified.")


Train       : 59,958 ZIP members
Validation  : 9,904 ZIP members
Test        : 9,927 ZIP members


,Split,Metadata_Records,Missing_Images
0,Train,59958,0
1,Validation,9904,0
2,Test,9927,0


PASS — Every referenced image exists in the appropriate ZIP.
No ZIP archive was copied or modified.


## 5. Verify captions

In [8]:
# ============================================================
# CELL 7 — CAPTION VALIDATION (with documented exclusion)
# ============================================================

caption_audit_rows = []
excluded_ids = {}

for split in ["Train", "Validation", "Test"]:
    df = captions[split]

    caption_col = (
        "Caption" if "Caption" in df.columns
        else "caption" if "caption" in df.columns
        else None
    )

    if caption_col is None:
        raise ValueError(
            f"{split}: no Caption/caption column found. "
            f"Columns: {list(df.columns)}"
        )

    is_missing = df[caption_col].isna()
    is_empty = (
        df[caption_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )
    bad_mask = is_missing | is_empty

    n_missing = int(is_missing.sum())
    n_empty = int((is_empty & ~is_missing).sum())

    if bad_mask.any():
        bad_ids = df.loc[bad_mask, "ID"].astype(str).tolist()
        excluded_ids[split] = bad_ids
        print(
            f"WARNING — {split}: excluding {len(bad_ids)} record(s) with "
            f"missing/empty captions (NaN={n_missing}, empty={n_empty}). "
            f"IDs: {bad_ids}"
        )
        # Drop from BOTH captions and concepts so downstream merges stay consistent
        captions[split] = df.loc[~bad_mask].reset_index(drop=True)
        concepts[split] = concepts[split][
            ~concepts[split]["ID"].astype(str).isin(bad_ids)
        ].reset_index(drop=True)

    caption_audit_rows.append({
        "Split": split,
        "Missing_Captions": n_missing,
        "Empty_Captions": n_empty,
        "Excluded_IDs": excluded_ids.get(split, []),
        "Remaining_Records": len(captions[split]),
    })

caption_audit_df = pd.DataFrame(caption_audit_rows)
display(caption_audit_df)

# Save the exclusion record — required for reproducibility.json / thesis appendix
exclusion_log_path = PREPARED_DIR / "excluded_empty_caption_records.json"
with open(exclusion_log_path, "w", encoding="utf-8") as f:
    json.dump(excluded_ids, f, indent=2)
print("Saved exclusion log:", exclusion_log_path)

print("PASS — Captions validated; empty/missing records excluded and logged.")

WARNING — Validation: excluding 1 record(s) with missing/empty captions (NaN=0, empty=1). IDs: ['ROCOv2_2023_valid_003902']


,Split,Missing_Captions,Empty_Captions,Excluded_IDs,Remaining_Records
0,Train,0,0,[],59958
1,Validation,0,1,[ROCOv2_2023_valid_003902],9903
2,Test,0,0,[],9927


Saved exclusion log: /content/drive/MyDrive/ROCOv2/Prepared/excluded_empty_caption_records.json
PASS — Captions validated; empty/missing records excluded and logged.


## 6. Verify CUI/concept information

In [9]:

# ============================================================
# CELL 8 — CUI / CONCEPT AUDIT
# ============================================================

def parse_cuis(value):
    if pd.isna(value):
        return []
    value = str(value).strip()
    if not value:
        return []
    return [x.strip() for x in value.split(";") if x.strip()]

parsed_cuis = {}

for split in ["Train", "Validation", "Test"]:
    parsed_cuis[split] = concepts[split]["CUIs"].apply(parse_cuis)

    empty_rows = int(parsed_cuis[split].map(len).eq(0).sum())

    if empty_rows:
        raise AssertionError(
            f"{split}: {empty_rows} records have no CUI annotation."
        )

observed_cuis = set(
    cui
    for split in parsed_cuis
    for row in parsed_cuis[split]
    for cui in row
)

mapping_cuis = set(
    cui_mapping_df["CUI"].astype(str).str.strip()
)

unmapped_cuis = sorted(observed_cuis - mapping_cuis)

print("Observed unique CUIs:", len(observed_cuis))
print("Mapped CUIs:", len(observed_cuis & mapping_cuis))
print("Unmapped CUIs:", len(unmapped_cuis))

if unmapped_cuis:
    print("WARNING — CUIs absent from cui_mapping.csv:")
    print(unmapped_cuis[:50])
else:
    print("PASS — All observed CUIs are represented in the CUI mapping.")


Observed unique CUIs: 1947
Mapped CUIs: 1934
Unmapped CUIs: 13
WARNING — CUIs absent from cui_mapping.csv:
['C0003486', 'C0006142', 'C0006826', 'C0008034', 'C0018946', 'C0153676', 'C0302148', 'C0333307', 'C0494165', 'C0521158', 'C0747635', 'C1266909', 'C2239176']


In [11]:
unmapped_report_path = PREPARED_DIR / "unmapped_cuis_report.json"
with open(unmapped_report_path, "w", encoding="utf-8") as f:
    json.dump({
        "observed_unique_cuis": len(observed_cuis),
        "mapped_cuis": len(observed_cuis & mapping_cuis),
        "unmapped_cuis": unmapped_cuis,
        "note": "None of these fall within the 18-label modality/anatomy taxonomy used for training.",
    }, f, indent=2)
print("Saved:", unmapped_report_path)

Saved: /content/drive/MyDrive/ROCOv2/Prepared/unmapped_cuis_report.json


In [10]:
EXPECTED_TOTAL = sum(len(captions[split]) for split in ["Train", "Validation", "Test"])
print("EXPECTED_TOTAL (post-exclusion):", EXPECTED_TOTAL)  # should print 79788

EXPECTED_TOTAL (post-exclusion): 79788


## 7. Construct the final verified 18-label taxonomy

In [12]:

# ============================================================
# CELL 9 — VERIFIED 18-LABEL TAXONOMY
# ============================================================

LABEL_TAXONOMY = {
    "C0000726": "Abdomen",
    "C0002978": "Angiogram",
    "C0006141": "Breast",
    "C0023216": "Lower Extremity",
    "C0024485": "Magnetic Resonance Imaging",
    "C0030797": "Pelvis",
    "C0032743": "Positron-Emission Tomography",
    "C0037303": "Bone structure of cranium",
    "C0037949": "Vertebral column",
    "C0040405": "X-Ray Computed Tomography",
    "C0041618": "Ultrasonography",
    "C0205106": "Transverse",
    "C0205129": "Sagittal",
    "C0817096": "Chest",
    "C1140618": "Upper Extremity",
    "C1306645": "Plain x-ray",
    "C1996865": "Postero-Anterior",
    "C1999039": "Anterior-Posterior",
}

assert len(LABEL_TAXONOMY) == 18

canonical_lookup = dict(
    zip(
        cui_mapping_df["CUI"].astype(str).str.strip(),
        cui_mapping_df["Canonical name"].astype(str)
    )
)

missing_mapping = set(LABEL_TAXONOMY) - set(canonical_lookup)
missing_observed = set(LABEL_TAXONOMY) - observed_cuis

if missing_mapping:
    raise AssertionError(
        "18-label taxonomy contains CUIs absent from cui_mapping.csv: "
        f"{sorted(missing_mapping)}"
    )

if missing_observed:
    raise AssertionError(
        "18-label taxonomy contains CUIs not observed in raw ROCOv2 data: "
        f"{sorted(missing_observed)}"
    )

label_taxonomy_df = pd.DataFrame([
    {
        "Index": i,
        "CUI": cui,
        "Label": label,
        "Canonical_Name": canonical_lookup[cui],
    }
    for i, (cui, label) in enumerate(LABEL_TAXONOMY.items())
])

display(label_taxonomy_df)

print("PASS — All 18 taxonomy CUIs occur in raw data and official mapping.")


,Index,CUI,Label,Canonical_Name
0,0,C0000726,Abdomen,Abdomen
1,1,C0002978,Angiogram,angiogram
2,2,C0006141,Breast,Breast
3,3,C0023216,Lower Extremity,Lower Extremity
4,4,C0024485,Magnetic Resonance Imaging,Magnetic Resonance Imaging
5,5,C0030797,Pelvis,Pelvis
6,6,C0032743,Positron-Emission Tomography,Positron-Emission Tomography
7,7,C0037303,Bone structure of cranium,Bone structure of cranium
8,8,C0037949,Vertebral column,Vertebral column
9,9,C0040405,X-Ray Computed Tomography,X-Ray Computed Tomography


PASS — All 18 taxonomy CUIs occur in raw data and official mapping.


## 8. Construct multi-hot label vectors

In [13]:

# ============================================================
# CELL 10 — MULTI-HOT TARGET CONSTRUCTION
# ============================================================

LABEL_COLUMNS = [
    f"label_{cui}"
    for cui in LABEL_TAXONOMY
]

def create_prepared_split(split):
    cap_df = captions[split].copy()
    cui_df = concepts[split].copy()

    caption_col = (
        "Caption" if "Caption" in cap_df.columns
        else "caption"
    )

    merged = cap_df[["ID", caption_col]].merge(
        cui_df[["ID", "CUIs"]],
        on="ID",
        how="inner",
        validate="one_to_one",
    )

    merged["ID"] = merged["ID"].astype(str).str.strip()

    merged["Image_Member"] = merged["ID"].apply(
        lambda x: resolve_image_member(x, split)
    )

    merged = merged.rename(
        columns={caption_col: "Caption"}
    )

    merged["CUI_String"] = merged["CUIs"].fillna("").astype(str)

    for cui in LABEL_TAXONOMY:
        merged[f"label_{cui}"] = merged["CUIs"].apply(
            lambda x, target=cui:
                int(target in parse_cuis(x))
        )

    return merged[
        ["ID", "Image_Member", "Caption", "CUI_String"]
        + LABEL_COLUMNS
    ]

prepared = {
    split: create_prepared_split(split)
    for split in ["Train", "Validation", "Test"]
}

for split, df in prepared.items():
    print(f"{split:12s}: {df.shape}")


Train       : (59958, 22)
Validation  : (9903, 22)
Test        : (9927, 22)


In [16]:
for split, df in prepared.items():
    n_missing_image = df["Image_Member"].isna().sum()
    print(f"{split}: {n_missing_image} unresolved image members")

Train: 0 unresolved image members
Validation: 0 unresolved image members
Test: 0 unresolved image members


## 9. Handle missing/rare labels transparently

In [14]:

# ============================================================
# CELL 11 — LABEL COUNTS AND RARE-LABEL AUDIT
# ============================================================

label_statistics_rows = []

for split in ["Train", "Validation", "Test"]:
    df = prepared[split]

    for cui, label in LABEL_TAXONOMY.items():
        count = int(df[f"label_{cui}"].sum())

        label_statistics_rows.append({
            "Split": split,
            "CUI": cui,
            "Label": label,
            "Count": count,
            "Percentage": 100.0 * count / len(df),
            "Rare_Below_Threshold": count < RARE_LABEL_THRESHOLD,
        })

label_statistics_df = pd.DataFrame(label_statistics_rows)

display(
    label_statistics_df.sort_values(
        ["Split", "Count"],
        ascending=[True, False]
    )
)

rare_labels_df = label_statistics_df[
    label_statistics_df["Rare_Below_Threshold"]
].copy()

print(
    f"Rare-label reporting threshold: < {RARE_LABEL_THRESHOLD} "
    "samples within a split."
)

if len(rare_labels_df):
    print("Rare labels are reported only; they are NOT silently removed.")
    display(rare_labels_df)
else:
    print("No label falls below the configured reporting threshold.")


,Split,CUI,Label,Count,Percentage,Rare_Below_Threshold
45,Test,C0040405,X-Ray Computed Tomography,3621,36.476277,False
51,Test,C1306645,Plain x-ray,2724,27.440314,False
46,Test,C0041618,Ultrasonography,1596,16.077365,False
40,Test,C0024485,Magnetic Resonance Imaging,1426,14.364864,False
49,Test,C0817096,Chest,1345,13.548907,False
53,Test,C1999039,Anterior-Posterior,1253,12.622142,False
36,Test,C0000726,Abdomen,565,5.691548,False
41,Test,C0030797,Pelvis,521,5.248313,False
52,Test,C1996865,Postero-Anterior,465,4.684195,False
39,Test,C0023216,Lower Extremity,458,4.613680,False


Rare-label reporting threshold: < 10 samples within a split.
Rare labels are reported only; they are NOT silently removed.


,Split,CUI,Label,Count,Percentage,Rare_Below_Threshold
47,Test,C0205106,Transverse,9,0.090662,True


## 10. Validate label mappings

In [17]:
# ============================================================
# CELL 12 — LABEL AND PREPARED-DATA ASSERTIONS (with zero-label exclusion)
# ============================================================

assert len(LABEL_TAXONOMY) == len(LABEL_COLUMNS)

zero_label_excluded_ids = {}

for split, df in prepared.items():

    required_columns = ["ID", "Image_Member", "Caption", "CUI_String"] + LABEL_COLUMNS
    missing_columns = set(required_columns) - set(df.columns)
    assert not missing_columns, f"{split}: missing required columns: {sorted(missing_columns)}"

    if df[required_columns].isna().any().any():
        raise AssertionError(f"{split}: NaN values found in required fields.")

    if not df["ID"].is_unique:
        raise AssertionError(f"{split}: duplicate sample IDs detected.")

    for col in LABEL_COLUMNS:
        values = set(df[col].unique())
        if not values.issubset({0, 1}):
            raise AssertionError(f"{split}: {col} contains non-binary values: {values}")

    zero_mask = df[LABEL_COLUMNS].sum(axis=1).eq(0)
    n_zero = int(zero_mask.sum())

    if n_zero:
        excluded_ids = df.loc[zero_mask, "ID"].astype(str).tolist()
        zero_label_excluded_ids[split] = excluded_ids
        print(
            f"WARNING — {split}: excluding {n_zero} sample(s) with no CUI "
            f"matching the 18-label taxonomy."
        )
        prepared[split] = df.loc[~zero_mask].reset_index(drop=True)

print("PASS — All prepared labels are binary.")
print("PASS — No NaN values in required fields.")
print("PASS — Sample IDs unique within each split.")

zero_label_log_path = PREPARED_DIR / "excluded_zero_label_records.json"
with open(zero_label_log_path, "w", encoding="utf-8") as f:
    json.dump(zero_label_excluded_ids, f, indent=2)
print("Saved exclusion log:", zero_label_log_path)

WARNING — Train: excluding 292 sample(s) with no CUI matching the 18-label taxonomy.
WARNING — Validation: excluding 84 sample(s) with no CUI matching the 18-label taxonomy.
WARNING — Test: excluding 57 sample(s) with no CUI matching the 18-label taxonomy.
PASS — All prepared labels are binary.
PASS — No NaN values in required fields.
PASS — Sample IDs unique within each split.
Saved exclusion log: /content/drive/MyDrive/ROCOv2/Prepared/excluded_zero_label_records.json


In [18]:
EXPECTED_TOTAL = sum(len(prepared[split]) for split in ["Train", "Validation", "Test"])
print("EXPECTED_TOTAL (post-exclusions):", EXPECTED_TOTAL)  # should print 79355

EXPECTED_TOTAL (post-exclusions): 79355


In [19]:
import shutil
from datetime import datetime

backup_dir = PREPARED_DIR / f"_archived_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
backup_dir.mkdir(parents=True, exist_ok=True)

for name in ["train_prepared.csv", "validation_prepared.csv", "test_prepared.csv",
             "label_mapping.json", "dataset_config.json", "prepared_statistics.csv",
             "reproducibility.json"]:
    src = PREPARED_DIR / name
    if src.exists():
        shutil.move(str(src), str(backup_dir / name))

print("Archived old artifacts to:", backup_dir)

Archived old artifacts to: /content/drive/MyDrive/ROCOv2/Prepared/_archived_20260819_105512


## 11. Generate and save prepared metadata

In [20]:

# ============================================================
# CELL 13 — SAVE PREPARED CSV FILES
# ============================================================

prepared_paths = {
    "Train": PREPARED_DIR / "train_prepared.csv",
    "Validation": PREPARED_DIR / "validation_prepared.csv",
    "Test": PREPARED_DIR / "test_prepared.csv",
}

# Protect existing experimental artifacts.
ALLOW_OVERWRITE = False

for split, path in prepared_paths.items():

    if path.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError(
            f"Refusing to overwrite existing prepared artifact:\n{path}\n\n"
            "This protection prevents accidental changes to the fixed "
            "experimental dataset. Remove/rename the old artifact or "
            "explicitly set ALLOW_OVERWRITE=True after review."
        )

    prepared[split].to_csv(path, index=False)
    print("Saved:", path)


Saved: /content/drive/MyDrive/ROCOv2/Prepared/train_prepared.csv
Saved: /content/drive/MyDrive/ROCOv2/Prepared/validation_prepared.csv
Saved: /content/drive/MyDrive/ROCOv2/Prepared/test_prepared.csv


In [21]:

# ============================================================
# CELL 14 — SAVE AUTHORITATIVE LABEL_MAPPING.JSON
# ============================================================

label_mapping = {
    cui: index
    for index, cui in enumerate(LABEL_TAXONOMY.keys())
}

label_mapping_path = PREPARED_DIR / "label_mapping.json"

if label_mapping_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing label_mapping.json:\n"
        f"{label_mapping_path}\n\n"
        "The label order must remain fixed after Notebook 02."
    )

with open(label_mapping_path, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2)

assert len(label_mapping) == len(LABEL_COLUMNS)

print("Saved:", label_mapping_path)
print("Authoritative label order:")
for cui, index in label_mapping.items():
    print(f"{index:02d} -> {cui} -> {LABEL_TAXONOMY[cui]}")


Saved: /content/drive/MyDrive/ROCOv2/Prepared/label_mapping.json
Authoritative label order:
00 -> C0000726 -> Abdomen
01 -> C0002978 -> Angiogram
02 -> C0006141 -> Breast
03 -> C0023216 -> Lower Extremity
04 -> C0024485 -> Magnetic Resonance Imaging
05 -> C0030797 -> Pelvis
06 -> C0032743 -> Positron-Emission Tomography
07 -> C0037303 -> Bone structure of cranium
08 -> C0037949 -> Vertebral column
09 -> C0040405 -> X-Ray Computed Tomography
10 -> C0041618 -> Ultrasonography
11 -> C0205106 -> Transverse
12 -> C0205129 -> Sagittal
13 -> C0817096 -> Chest
14 -> C1140618 -> Upper Extremity
15 -> C1306645 -> Plain x-ray
16 -> C1996865 -> Postero-Anterior
17 -> C1999039 -> Anterior-Posterior


## 12. Save dataset configuration, statistics, and reproducibility record

In [22]:

# ============================================================
# CELL 15 — SAVE DATASET_CONFIG.JSON
# ============================================================

dataset_config = {
    "dataset": "ROCOv2",
    "notebook": "02_ROCOv2_Dataset_Preparation",
    "random_seed": SEED,
    "split_protocol": SPLIT_PROTOCOL,
    "label_count": len(label_mapping),
    "label_columns": LABEL_COLUMNS,
    "label_mapping_file": "label_mapping.json",
    "image_storage": "Original ROCOv2 ZIP archives",
    "image_copy_performed": False,
    "later_notebook_image_access": "Read directly from original ZIP files",
    "test_usage": "Untouched final evaluation; never used for training/model selection/tuning",
    "rare_label_threshold": RARE_LABEL_THRESHOLD,
}

dataset_config_path = PREPARED_DIR / "dataset_config.json"

if dataset_config_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing dataset_config.json:\n"
        f"{dataset_config_path}"
    )

with open(dataset_config_path, "w", encoding="utf-8") as f:
    json.dump(dataset_config, f, indent=2)

print("Saved:", dataset_config_path)

# Prepared statistics
prepared_statistics_df = pd.DataFrame([
    {
        "Split": split,
        "Records": len(df),
        "Unique_IDs": df["ID"].nunique(),
        "Images_Resolved": int(df["Image_Member"].notna().sum()),
        "Nonempty_Captions": int(
            df["Caption"].fillna("").astype(str).str.strip().ne("").sum()
        ),
    }
    for split, df in prepared.items()
])

prepared_statistics_path = PREPARED_DIR / "prepared_statistics.csv"

if prepared_statistics_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing prepared_statistics.csv:\n"
        f"{prepared_statistics_path}"
    )

prepared_statistics_df.to_csv(
    prepared_statistics_path,
    index=False
)

display(prepared_statistics_df)
print("Saved:", prepared_statistics_path)


Saved: /content/drive/MyDrive/ROCOv2/Prepared/dataset_config.json


,Split,Records,Unique_IDs,Images_Resolved,Nonempty_Captions
0,Train,59666,59666,59666,59666
1,Validation,9819,9819,9819,9819
2,Test,9870,9870,9870,9870


Saved: /content/drive/MyDrive/ROCOv2/Prepared/prepared_statistics.csv


In [23]:

# ============================================================
# CELL 16 — SAVE REPRODUCIBILITY.JSON
# ============================================================

reproducibility = {
    "environment": ENVIRONMENT,
    "random_seed": SEED,
    "split_protocol": SPLIT_PROTOCOL,
    "label_mapping": label_mapping,
    "label_taxonomy": LABEL_TAXONOMY,
    "hyperparameters": {
        "rare_label_threshold": RARE_LABEL_THRESHOLD,
    },
    "dataset_statistics": prepared_statistics_df.to_dict(
        orient="records"
    ),
}

reproducibility_path = PREPARED_DIR / "reproducibility.json"

if reproducibility_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing reproducibility.json:\n"
        f"{reproducibility_path}"
    )

with open(reproducibility_path, "w", encoding="utf-8") as f:
    json.dump(
        reproducibility,
        f,
        indent=2,
        default=str
    )

print("Saved:", reproducibility_path)


Saved: /content/drive/MyDrive/ROCOv2/Prepared/reproducibility.json


## 13. Final integrity checks

In [24]:

# ============================================================
# CELL 17 — FINAL INTEGRITY CHECKS
# ============================================================

# Re-read authoritative label mapping from disk.
with open(label_mapping_path, "r", encoding="utf-8") as f:
    saved_label_mapping = json.load(f)

assert len(saved_label_mapping) == len(LABEL_COLUMNS)

saved_label_columns = [
    f"label_{cui}"
    for cui, _ in sorted(
        saved_label_mapping.items(),
        key=lambda item: item[1]
    )
]

assert saved_label_columns == LABEL_COLUMNS, (
    "Saved label_mapping.json does not reproduce the expected label order."
)

# Re-load prepared CSVs and validate them independently.
for split, path in prepared_paths.items():

    disk_df = pd.read_csv(path)

    assert len(disk_df) == len(prepared[split]), (
        f"{split}: saved row count differs from generated row count."
    )

    required = [
        "ID",
        "Image_Member",
        "Caption",
        "CUI_String",
    ] + LABEL_COLUMNS

    assert not disk_df[required].isna().any().any(), (
        f"{split}: NaN detected after reloading prepared CSV."
    )

    for col in LABEL_COLUMNS:
        assert set(disk_df[col].unique()).issubset({0, 1}), (
            f"{split}: non-binary values found in {col}."
        )

# Verify split independence one final time.
saved_ids = {
    split: set(
        pd.read_csv(path)["ID"].astype(str).str.strip()
    )
    for split, path in prepared_paths.items()
}

assert not saved_ids["Train"] & saved_ids["Validation"]
assert not saved_ids["Train"] & saved_ids["Test"]
assert not saved_ids["Validation"] & saved_ids["Test"]

print("PASS — Prepared files reload successfully.")
print("PASS — Authoritative label mapping is reproducible from disk.")
print("PASS — All labels are binary.")
print("PASS — Required fields contain no NaN.")
print("PASS — Saved splits contain no ID overlap.")
print("PASS — Original image ZIPs were not copied or modified.")


PASS — Prepared files reload successfully.
PASS — Authoritative label mapping is reproducible from disk.
PASS — All labels are binary.
PASS — Required fields contain no NaN.
PASS — Saved splits contain no ID overlap.
PASS — Original image ZIPs were not copied or modified.


## 14. Final artifact manifest

In [25]:

# ============================================================
# CELL 18 — FINAL ARTIFACT MANIFEST
# ============================================================

artifact_paths = [
    PREPARED_DIR / "train_prepared.csv",
    PREPARED_DIR / "validation_prepared.csv",
    PREPARED_DIR / "test_prepared.csv",
    PREPARED_DIR / "label_mapping.json",
    PREPARED_DIR / "dataset_config.json",
    PREPARED_DIR / "prepared_statistics.csv",
    PREPARED_DIR / "reproducibility.json",
]

manifest = []

for path in artifact_paths:
    manifest.append({
        "Artifact": path.name,
        "Path": str(path),
        "Exists": path.exists(),
        "Size_MB": (
            round(path.stat().st_size / (1024 ** 2), 4)
            if path.exists()
            else None
        ),
    })

artifact_manifest_df = pd.DataFrame(manifest)
display(artifact_manifest_df)

missing = artifact_manifest_df.loc[
    ~artifact_manifest_df["Exists"],
    "Path"
].tolist()

if missing:
    raise FileNotFoundError(
        "Required Notebook 02 artifacts are missing:\n"
        + "\n".join(missing)
    )

print("\n" + "=" * 80)
print("NOTEBOOK 02 — DATASET PREPARATION COMPLETE")
print("=" * 80)

print("\nFILES CREATED:")
for path in artifact_paths:
    print("  ✓", path)

print("\nMODEL CHECKPOINT:")
print("  N/A — data-preparation notebook.")

print("\nMETRICS / STATISTICS:")
print("  ✓ prepared_statistics.csv")
print("  ✓ label frequency audit available in-memory and reproducibility record.")

print("\nFIGURES:")
print("  N/A — publication figures belong to EDA/model-analysis notebooks.")

print("\nTABLES:")
print("  ✓ train_prepared.csv")
print("  ✓ validation_prepared.csv")
print("  ✓ test_prepared.csv")
print("  ✓ prepared_statistics.csv")

print("\nLOGS / REPRODUCIBILITY:")
print("  ✓ reproducibility.json")
print("  ✓ dataset_config.json")
print("  ✓ label_mapping.json")

print("\nPROTOCOL:")
print("  ✓ Train = training")
print("  ✓ Validation = calibration/development")
print("  ✓ Test = untouched final evaluation")
print("  ✓ Original ROCOv2 ZIPs remain in place")
print("  ✓ Later notebooks must load label_mapping.json")
print("  ✓ Later notebooks must read images directly from original ZIPs")

print("=" * 80)


,Artifact,Path,Exists,Size_MB
0,train_prepared.csv,/content/drive/MyDrive/ROCOv2/Prepared/train_p...,True,15.2787
1,validation_prepared.csv,/content/drive/MyDrive/ROCOv2/Prepared/validat...,True,2.6118
2,test_prepared.csv,/content/drive/MyDrive/ROCOv2/Prepared/test_pr...,True,2.6081
3,label_mapping.json,/content/drive/MyDrive/ROCOv2/Prepared/label_m...,True,0.0003
4,dataset_config.json,/content/drive/MyDrive/ROCOv2/Prepared/dataset...,True,0.0010
5,prepared_statistics.csv,/content/drive/MyDrive/ROCOv2/Prepared/prepare...,True,0.0001
6,reproducibility.json,/content/drive/MyDrive/ROCOv2/Prepared/reprodu...,True,0.0021



NOTEBOOK 02 — DATASET PREPARATION COMPLETE

FILES CREATED:
  ✓ /content/drive/MyDrive/ROCOv2/Prepared/train_prepared.csv
  ✓ /content/drive/MyDrive/ROCOv2/Prepared/validation_prepared.csv
  ✓ /content/drive/MyDrive/ROCOv2/Prepared/test_prepared.csv
  ✓ /content/drive/MyDrive/ROCOv2/Prepared/label_mapping.json
  ✓ /content/drive/MyDrive/ROCOv2/Prepared/dataset_config.json
  ✓ /content/drive/MyDrive/ROCOv2/Prepared/prepared_statistics.csv
  ✓ /content/drive/MyDrive/ROCOv2/Prepared/reproducibility.json

MODEL CHECKPOINT:
  N/A — data-preparation notebook.

METRICS / STATISTICS:
  ✓ prepared_statistics.csv
  ✓ label frequency audit available in-memory and reproducibility record.

FIGURES:
  N/A — publication figures belong to EDA/model-analysis notebooks.

TABLES:
  ✓ train_prepared.csv
  ✓ validation_prepared.csv
  ✓ test_prepared.csv
  ✓ prepared_statistics.csv

LOGS / REPRODUCIBILITY:
  ✓ reproducibility.json
  ✓ dataset_config.json
  ✓ label_mapping.json

PROTOCOL:
  ✓ Train = trainin